# Gate Modulation Regression Analysis — Age and Tissue-Dependent Pathway Representations

This notebook provides **statistically defensible evidence** that the gating
mechanism learns age-dependent and tissue-dependent biological representations.
It replaces the previous "shared age-predictive" vs "tissue-specific" framework
with a principled regression-based approach.

**Two questions:**
1. Which pathway-associated gates change with age **overall**?
2. Which pathway-associated gates exhibit **different age relationships across
   tissues**?

**Gate value definition.** A gating value of 1 indicates no modulation of the
corresponding neuron output. Values greater than 1 indicate **amplification** of
the neuron representation, whereas values below 1 indicate **attenuation**.

**Statistical Analysis Plan.**

| Analysis | Model | Purpose |
|---|---|---|
| 1 | `Gate ~ Age + Tissue + Study` | Overall age-associated gates (common slope) |
| 2 | `Gate ~ Age * Tissue + Study` | Tissue-dependent modulation (interaction test) |

Study = `Accession_Code` (GSE dataset identifier), included as a categorical
fixed effect to control for batch/study differences.

**Figure structure.**

| Panel | Content | Selection |
|---|---|---|
| A1 | Coefficient plot: standardized age coefficients with 95% CI | FDR < 0.05, capped at 20 (top 10 pos + top 10 neg by \|std β\|) |
| A2 | 4 representative raw-scale gate-vs-age trajectories | 2 largest positive Δg(20→90) + 2 largest negative Δg(20→90) |
| B | Tissue-dependent age associations (interaction test) | Analysis 2 top N (interaction FDR × effect size) |

**Panel A notes.** The additive model (Gate ~ Age + Tissue + Study) estimates a
**common age slope** across tissues. Colored lines in trajectory plots are
tissue-specific predictions under a common age coefficient; vertical separation
reflects adjusted tissue intercepts, not tissue-specific age slopes. Tissue
dependence requires the separate Age × Tissue interaction test (Panel B).
Panel A uses Δg(20→90) because the pooled data span approximately ages 18–89.

**Coefficient direction vs gating state.** A positive age coefficient
indicates **increasing amplification with age** (if the representation is above
1 on average) or **decreasing attenuation with age** (if below 1). A negative
age coefficient indicates **decreasing amplification with age** (if above 1) or
**increasing attenuation with age** (if below 1). Coefficient direction and
gating state are separate concepts — a representation can be amplified
throughout life yet show a negative age slope (becoming less amplified).

**Interpretation.** Most overall age-associated representations remained
amplified (gate > 1) across the observed age range, with aging producing modest
incremental changes in the degree of amplification. The magnitude of raw-scale
changes varies among representations, ranging from minimal shifts (~0.006 over
70 years) to more pronounced age gradients (~0.22 over 70 years). Statistical
certainty (FDR) is not equivalent to substantial modulation — Blood's large
sample size (n=4105) drives very small FDR values. A subset of pathway
representations exhibited statistically robust but generally modest
age-associated increases in amplification.

**Scope.** Restricted to the **8 healthy-tissue testing datasets** (blood,
liver, muscle, adipose). Disease/disease-adjacent cohorts are excluded.
After filtering, 4 organ groups remain: Blood, Adipose, Muscle, Liver.

**Caveat.** Blood contains substantially more samples (n=4105) than the other
tissues (Adipose n=142, Muscle n=108, Liver n=76). Significance differences
across tissues should not be over-interpreted as biological specificity.

**Interpretation language.** This notebook measures **gate values**, not
pathway activity. We use: *amplification of pathway representations*,
*attenuation of pathway representations*, *age-dependent gating*, *modulation
of pathway-associated neurons*, *model reliance on pathway representations*.
We do NOT use: pathway activation, biological upregulation, pathway suppression.

**Config.** Set `STAGE_INDICES` to the gated stages of interest. Pathway
display names are loaded from `edges.tsv`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import seaborn as sns
import statsmodels.api as sm
import torch
from statsmodels.stats.multitest import multipletests

%matplotlib inline

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 6.5,
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
})
sns.set_style("white")

# === Configuration ===
# Anchor all paths to the repo root so the notebook runs from any kernel cwd.
_REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
GATES_DIR = _REPO_ROOT / "outputs" / "gates"
OUTPUT_DIR = _REPO_ROOT / "outputs" / "figures"
FOLD = "0"
SPLIT = "testing"

STAGE_INDICES = [0, 1, 2, 3]  # input_to_gene, gene_to_l2, l2_to_l1, l1_to_l0
GROUP_COL = "organ"
MIN_SAMPLES_PER_GROUP = 5
FDR_ALPHA = 0.05

# Figure parameters
N_OVERALL = 4        # top gates for Row A trajectories (Analysis 1)
N_COEF_PLOT_MAX = 20  # max features in Panel A1 coefficient plot
AGE_RANGE_FOR_DELTA = (20, 90)  # age range for delta_g computation
STAGE_LABELS = {
    "input_to_gene": "input-to-gene",
    "gene_to_l2": "gene-to-pathway",
    "l2_to_l1": "L2-to-L1 pathway",
    "l1_to_l0": "higher-order pathway",
}
A1_XAXIS_MODE = "two_panels"  # "single", "two_panels", or "broken"

# Panel A font sizes (adjust for readability)
A1_TITLE_FONT = 12       # A1 coefficient plot title
A1_LABEL_FONT = 10      # A1 y-axis pathway labels
A1_XLABEL_FONT = 10      # A1 x-axis labels
A2_TITLE_FONT = 10       # A2 trajectory subplot titles
A2_LABEL_FONT = 10       # A2 axis labels (ylabel, xlabel)
A2_LEGEND_FONT = 10       # A2 legend
N_INTERACTION = 5    # top gates for Row B (Analysis 2)
BLOOD_SCATTER_MAX = 500  # downsample Blood scatter points
BASE_FONT = 14

TISSUE_COLORS = {
    "Blood": "#d62728",
    "Adipose": "#ff7f0e",
    "Muscle": "#2ca02c",
    "Liver": "#1f77b4",
}
TISSUE_ORDER = ["Blood", "Adipose", "Muscle", "Liver"]

EDGES_TSV = _REPO_ROOT / "resources" / "masks" / "edges.tsv"

# === Load gate data ===
gate_file = GATES_DIR / f"fold_{FOLD}_{SPLIT}_gates.pt"
data = torch.load(gate_file, weights_only=False)

# === Filter to healthy-tissue testing datasets ===
HEALTHY_TESTING_DATASETS = [
    "GSE55763", "GSE210255",  # Blood
    "GSE48325", "GSE61258",   # Liver
    "GSE61452", "GSE50498",   # Muscle
    "GSE61450", "GSE61453",   # Adipose
]

_n_before = len(data["metadata"])
_accession_col = data["metadata"]["Accession_Code"].values
_healthy_mask = np.isin(_accession_col, HEALTHY_TESTING_DATASETS)

data["metadata"] = data["metadata"].loc[_healthy_mask].reset_index(drop=True)
data["labels_raw"] = data["labels_raw"][_healthy_mask]
data["labels"] = data["labels"][_healthy_mask]
data["dataset_ids"] = data["dataset_ids"][_healthy_mask]
data["accession_codes"] = [c for c, m in zip(data["accession_codes"], _healthy_mask) if m]

for _sd in data["stages"].values():
    if _sd.get("gate") is not None:
        for _key in ("gate", "value", "gate_mean", "gate_std", "value_mean"):
            _sd[_key] = _sd[_key][_healthy_mask]

_n_after = len(data["metadata"])
_excluded = sorted(set(_accession_col) - set(HEALTHY_TESTING_DATASETS))
print(f"Filtered to healthy-tissue datasets: {_n_before} -> {_n_after} samples")
print(f"Excluded {len(_excluded)} datasets: {', '.join(_excluded)}")

metadata = data["metadata"].copy()
labels_raw = data["labels_raw"].numpy()

# Valid groups (tissues with >= MIN_SAMPLES_PER_GROUP)
organ_counts = metadata[GROUP_COL].value_counts()
valid_groups = organ_counts[organ_counts >= MIN_SAMPLES_PER_GROUP].index.tolist()
print(f"\n{GROUP_COL} counts:")
print(organ_counts)
print(f"\nValid groups (>= {MIN_SAMPLES_PER_GROUP} samples): {len(valid_groups)}")

# Tissue x study cross-tab
print("\nTissue x study cross-tab:")
print(metadata.groupby([GROUP_COL, "Accession_Code"]).size().unstack(fill_value=0))

# === Load pathway names from edges.tsv ===
name_map = {}
if EDGES_TSV.exists():
    edges = pd.read_csv(EDGES_TSV, sep="\t", low_memory=False)
    for stage in ["gene_to_l2", "l2_to_l1", "l1_to_l0"]:
        stage_edges = edges[edges["hidden_layer"] == stage]
        for _, row in stage_edges.iterrows():
            if pd.notna(row.get("target_name")) and str(row["target_name"]).strip():
                name_map[row["target"]] = str(row["target_name"]).strip()
            if pd.notna(row.get("source_name")) and str(row["source_name"]).strip():
                name_map[row["source"]] = str(row["source_name"]).strip()


def short_name(fid, max_len=30):
    full = name_map.get(fid, fid)
    return full[:max_len] + "…" if len(full) > max_len else full


def wrap_name(fid, max_len=22):
    """Wrap pathway name at a space near the middle (two lines)."""
    full = name_map.get(fid, fid)
    if len(full) <= max_len:
        return full
    mid = len(full) // 2
    for offset in range(mid):
        if mid + offset < len(full) and full[mid + offset] == " ":
            return full[: mid + offset] + "\n" + full[mid + offset + 1 :]
        if mid - offset >= 0 and full[mid - offset] == " ":
            return full[: mid - offset] + "\n" + full[mid - offset + 1 :]
    return full[:max_len] + "\n" + full[max_len:]


def format_fdr(fdr_p, label="FDR"):
    """Format FDR/q-value, replacing numerical-zero with a lower bound."""
    if fdr_p < 1e-299:
        return f"{label} < 1e-299"
    return f"{label}={fdr_p:.2e}"


# === Helper: build per-stage gate DataFrame ===
def build_gate_df(stage_idx):
    sd = data["stages"][stage_idx]
    gm = sd["gate_mean"]
    fn = sd["feature_names"]
    sn = sd["name"]
    _N, _D = gm.shape
    cols = fn if fn else [f"f{i}" for i in range(_D)]
    df = pd.DataFrame(gm.numpy(), columns=cols)
    df["age"] = labels_raw
    df[GROUP_COL] = metadata[GROUP_COL].values
    df["Accession_Code"] = metadata["Accession_Code"].values
    feat_cols = [c for c in cols if c in df.columns]
    return df, feat_cols, sn


print(f"\nPathway names loaded: {len(name_map)}")
print(f"\nStage indices: {STAGE_INDICES}")
for si in STAGE_INDICES:
    _df, _fc, _sn = build_gate_df(si)
    print(f"  Stage {si}: {_sn}, {len(_fc)} features, {_df.shape[0]} samples")

In [ ]:
# === Analysis 1: Overall Age-Associated Gates ===
# Model: Gate ~ Age + C(Tissue) + C(Study)  (additive OLS)
# Identifies gates with age-dependent modulation after accounting for
# differences between tissues and studies.
#
# Outputs per feature: age coefficient, SE, t-stat, p-value, FDR-p,
#   standardized coefficient = raw_coef * sd(age) / sd(gate)
# Ranking: FDR significance -> |standardized coefficient|

analysis1 = {}  # stage_idx -> (df, gate_filt, feature_cols, stage_name)

for STAGE_IDX in STAGE_INDICES:
    gate_df, feature_cols, stage_name = build_gate_df(STAGE_IDX)
    gate_filt = gate_df[gate_df[GROUP_COL].isin(valid_groups)].copy()

    # Build additive design matrix once (same X for all features)
    template = gate_filt[["age", GROUP_COL, "Accession_Code"]].copy()
    X_add = patsy.dmatrix(
        f"1 + age + C({GROUP_COL}, Treatment(reference='Blood')) + C(Accession_Code)",
        data=template,
        return_type="dataframe",
    )
    add_cols = X_add.columns.tolist()
    age_pos = add_cols.index("age")
    X_add_np = X_add.values
    sd_age = np.std(template["age"])

    rows = []
    for feat in feature_cols:
        y = gate_filt[feat].values
        sd_y = np.std(y)
        if sd_y < 1e-12:
            rows.append({
                "feature": feat, "age_coef": 0.0, "age_se": np.nan,
                "age_t": 0.0, "age_p": 1.0, "std_coef": 0.0,
            })
            continue

        model = sm.OLS(y, X_add_np).fit()
        coef = model.params[age_pos]
        se = model.bse[age_pos]
        t = model.tvalues[age_pos]
        p = model.pvalues[age_pos]
        std_coef = coef * (sd_age / sd_y)

        rows.append({
            "feature": feat, "age_coef": coef, "age_se": se,
            "age_t": t, "age_p": p, "std_coef": std_coef,
        })

    df1 = pd.DataFrame(rows)
    _, fdr_p, _, _ = multipletests(
        df1["age_p"].values, alpha=FDR_ALPHA, method="fdr_bh"
    )
    df1["fdr_p"] = fdr_p
    df1["significant"] = fdr_p < FDR_ALPHA
    df1["abs_std_coef"] = df1["std_coef"].abs()
    _delta = AGE_RANGE_FOR_DELTA[1] - AGE_RANGE_FOR_DELTA[0]
    df1["delta_g_20_90"] = df1["age_coef"] * _delta
    df1["abs_delta_g"] = df1["delta_g_20_90"].abs()
    df1 = df1.sort_values(
        ["significant", "abs_std_coef"], ascending=[False, False]
    ).reset_index(drop=True)
    df1["rank"] = range(1, len(df1) + 1)

    analysis1[STAGE_IDX] = (df1, gate_filt, feature_cols, stage_name)

    n_sig = int(df1["significant"].sum())
    print(
        f"Stage {STAGE_IDX} ({stage_name}): "
        f"{n_sig}/{len(df1)} FDR-significant age-associated gates"
    )
    for _, r in df1.head(5).iterrows():
        print(
            f"  {r['rank']}. {r['feature']} ({short_name(r['feature'], 30)}): "
            f"coef={r['age_coef']:.6f}, std={r['std_coef']:.4f}, "
            f"\u0394g={r['delta_g_20_90']:.4f}, {format_fdr(r['fdr_p'])}"
        )

In [ ]:
# === Analysis 2: Tissue-Dependent Modulation ===
# Model: Gate ~ Age * C(Tissue) + C(Study)  (interaction OLS)
# Tests H0: all Age:Tissue interaction terms = 0  (omnibus Wald F-test)
#
# Outputs per feature: omnibus interaction F-stat, p-value, FDR-p,
#   tissue-specific slopes, interaction effect size (max-min of slopes)
# Ranking: FDR significance -> interaction effect size

analysis2 = {}  # stage_idx -> df

for STAGE_IDX in STAGE_INDICES:
    df1, gate_filt, feature_cols, stage_name = analysis1[STAGE_IDX]

    template = gate_filt[["age", GROUP_COL, "Accession_Code"]].copy()
    X_int = patsy.dmatrix(
        f"1 + age * C({GROUP_COL}, Treatment(reference='Blood')) + C(Accession_Code)",
        data=template,
        return_type="dataframe",
    )
    int_cols = X_int.columns.tolist()
    age_pos = int_cols.index("age")
    X_int_np = X_int.values

    # Identify interaction columns (contain ":")
    inter_cols = [c for c in int_cols if ":" in c]
    inter_indices = [int_cols.index(c) for c in inter_cols]

    # Parse tissue names from interaction column names
    # Format: "age:C(organ, Treatment(reference='Blood'))[T.Adipose]"
    tissue_inter = {}
    for c in inter_cols:
        tname = c.split("[T.")[1].rstrip("]")
        tissue_inter[tname] = int_cols.index(c)

    # Reference tissue = the one NOT in interaction terms
    ref_tissue = [t for t in valid_groups if t not in tissue_inter][0]

    rows = []
    for feat in feature_cols:
        y = gate_filt[feat].values
        if np.std(y) < 1e-12:
            row_dict = {
                "feature": feat, "omnibus_F": 0.0, "omnibus_p": 1.0,
                "interaction_effect_size": 0.0,
            }
            for t in valid_groups:
                row_dict[f"{t}_slope"] = 0.0
            rows.append(row_dict)
            continue

        model = sm.OLS(y, X_int_np).fit()

        # Omnibus F-test for interaction terms
        R = np.zeros((len(inter_indices), len(int_cols)))
        for j, idx in enumerate(inter_indices):
            R[j, idx] = 1
        ft = model.f_test(R)
        omnibus_p = float(np.asarray(ft.pvalue).item())
        fval = ft.fvalue
        omnibus_f = float(np.asarray(fval).item()) if fval is not None else 0.0

        # Tissue-specific slopes
        age_coef = model.params[age_pos]
        slopes = {ref_tissue: float(age_coef)}
        for tname, idx in tissue_inter.items():
            slopes[tname] = float(age_coef + model.params[idx])

        slope_vals = list(slopes.values())
        effect_size = max(slope_vals) - min(slope_vals)

        row_dict = {
            "feature": feat, "omnibus_F": omnibus_f, "omnibus_p": omnibus_p,
            "interaction_effect_size": effect_size,
        }
        for t in valid_groups:
            row_dict[f"{t}_slope"] = slopes.get(t, 0.0)
        rows.append(row_dict)

    df2 = pd.DataFrame(rows)
    _, fdr_p2, _, _ = multipletests(
        df2["omnibus_p"].values, alpha=FDR_ALPHA, method="fdr_bh"
    )
    df2["fdr_p"] = fdr_p2
    df2["significant"] = fdr_p2 < FDR_ALPHA
    df2 = df2.sort_values(
        ["significant", "interaction_effect_size"], ascending=[False, False]
    ).reset_index(drop=True)
    df2["rank"] = range(1, len(df2) + 1)

    analysis2[STAGE_IDX] = df2

    n_sig = int(df2["significant"].sum())
    print(
        f"Stage {STAGE_IDX} ({stage_name}): "
        f"{n_sig}/{len(df2)} FDR-significant tissue-dependent gates  "
        f"(ref tissue: {ref_tissue})"
    )
    for _, r in df2.head(5).iterrows():
        print(
            f"  {r['rank']}. {r['feature']} ({short_name(r['feature'], 30)}): "
            f"F={r['omnibus_F']:.2f}, FDR={r['fdr_p']:.2e}, "
            f"effect={r['interaction_effect_size']:.6f}"
        )

In [ ]:
# === Figure Panel A: Overall Age-Associated Pathway Representations ===
# Two-row composite figure per stage:
#   A1 (top): coefficient plot — standardized age coefficients with 95% CI,
#             capped at N_COEF_PLOT_MAX (top 10 pos + top 10 neg by |std beta|)
#   A2 (bottom): 4 representative raw-scale gate-vs-age trajectories
#               2 largest positive delta_g(20->90) + 2 largest negative
#
# The additive model (Gate ~ Age + Tissue + Study) estimates a COMMON age slope.
# Colored lines are tissue-specific predictions under a common age coefficient;
# vertical separation reflects adjusted tissue intercepts, not tissue-specific
# age slopes.

for STAGE_IDX in STAGE_INDICES:
    df1, gate_filt, feature_cols, stage_name = analysis1[STAGE_IDX]
    D_s = len(feature_cols)
    stage_label = STAGE_LABELS.get(stage_name, stage_name)

    df_sig = df1[df1["significant"]].copy()
    n_sig = len(df_sig)

    # --- A1: Coefficient plot data ---
    if n_sig <= N_COEF_PLOT_MAX:
        df_plot = df_sig.sort_values("std_coef")
        coef_note = f"{n_sig} of {D_s} tested (FDR < {FDR_ALPHA})"
    else:
        n_half = N_COEF_PLOT_MAX // 2
        df_pos = df_sig[df_sig["std_coef"] > 0].nlargest(n_half, "abs_std_coef")
        df_neg = df_sig[df_sig["std_coef"] < 0].nlargest(n_half, "abs_std_coef")
        df_plot = pd.concat([df_neg, df_pos]).sort_values("std_coef")
        coef_note = (
            f"{len(df_plot)} shown of {n_sig} significant / "
            f"{D_s} tested (FDR < {FDR_ALPHA})"
        )

    gate_means = {
        feat: float(gate_filt[feat].mean()) for feat in df_plot["feature"]
    }

    # --- A2: Trajectory selection (2 largest pos + 2 largest neg delta_g) ---
    selected_indices = []

    df_pos_dg = df_sig[df_sig["delta_g_20_90"] > 0].nlargest(2, "delta_g_20_90")
    df_neg_dg = df_sig[df_sig["delta_g_20_90"] < 0].nsmallest(2, "delta_g_20_90")
    selected_indices = list(df_pos_dg.index) + list(df_neg_dg.index)

    # Fill remaining from top by |delta_g| if < N_OVERALL
    if len(selected_indices) < N_OVERALL:
        for idx in df_sig.sort_values("abs_delta_g", ascending=False).index:
            if idx not in selected_indices:
                selected_indices.append(idx)
                if len(selected_indices) >= N_OVERALL:
                    break

    traj_rows = [df_sig.loc[idx] for idx in selected_indices[:N_OVERALL]]

    # --- Figure layout: 2 rows via gridspec ---
    _wrapped_preview = [wrap_name(f, 28) for f in df_plot["feature"]]
    _total_lines = sum(lbl.count("\n") + 1 for lbl in _wrapped_preview)
    coef_height = max(7.5, _total_lines * 0.22)
    fig = plt.figure(figsize=(15, coef_height + 4.5))
    gs = fig.add_gridspec(
        2, N_OVERALL,
        height_ratios=[coef_height, 3.5],
        hspace=0.05, wspace=0.30,
    )

    # === A1: Coefficient plot ===
    # Wrap pathway names before calculating vertical positions
    y_labels = [wrap_name(feat, 28) for feat in df_plot["feature"]]

    # Variable y-positions: multiline labels get extra space
    y_positions = []
    current_y = 0.0
    for label in y_labels:
        n_lines = label.count("\n") + 1
        y_positions.append(current_y)
        current_y += 1.0 + 0.62 * (n_lines - 1)

    # Map feature -> y-position for split-axis modes
    feat_to_ypos = {
        row["feature"]: y_positions[i]
        for i, (_, row) in enumerate(df_plot.iterrows())
    }

    df_neg = df_plot[df_plot["std_coef"] < 0]
    df_pos = df_plot[df_plot["std_coef"] >= 0]

    def _plot_coef_point(ax, row, y_pos):
        feat = row["feature"]
        std_c = row["std_coef"]
        t_val = row["age_t"]
        gm = gate_means.get(feat, 1.0)
        if abs(t_val) > 1e-12:
            std_se = abs(std_c / t_val)
            ci_half = 1.96 * std_se
        else:
            ci_half = 0.0
        color = "#d62728" if std_c > 0 else "#1f77b4"
        marker = "o" if gm > 1.0 else "D"
        mfc = color if gm > 1.0 else "white"
        ax.errorbar(
            std_c, y_pos, xerr=ci_half,
            fmt=marker, color=color, capsize=2, markersize=4,
            markerfacecolor=mfc, markeredgecolor=color,
            elinewidth=0.8, alpha=0.8,
        )

    if A1_XAXIS_MODE == "single" or len(df_neg) == 0 or len(df_pos) == 0:
        # ── Single full-width axis ──
        ax_coef = fig.add_subplot(gs[0, :])
        for i, (_, row) in enumerate(df_plot.iterrows()):
            _plot_coef_point(ax_coef, row, y_positions[i])
        ax_coef.set_yticks(y_positions)
        ax_coef.set_yticklabels(
            y_labels, fontsize=A1_LABEL_FONT,
            horizontalalignment="right", linespacing=0.92,
        )
        ax_coef.set_ylim(current_y - 0.4, -0.6)
        ax_coef.set_xlabel(
            "Standardized age coefficient (SD gate / SD age)",
            fontsize=A1_XLABEL_FONT,
        )
        ax_coef.set_title(
            f"Overall age-associated pathway representations "
            f"({coef_note})\n"
            "Red/blue = positive/negative age coefficient  |  "
            "Filled/open markers = mean gate above/below 1",
            fontsize=A1_TITLE_FONT,
        )
        ax_coef.axvline(
            x=0, color="gray", linestyle="--",
            alpha=0.5, linewidth=0.8,
        )
        ax_coef.tick_params(axis="y", labelsize=A1_LABEL_FONT, pad=6, length=0)
        ax_coef.tick_params(axis="x", labelsize=A1_XLABEL_FONT - 1)

    else:
        # ── Split axis: two panels (negative | positive) ──
        ax_neg = fig.add_subplot(gs[0, 0:2])
        ax_pos = fig.add_subplot(gs[0, 2:4], sharey=ax_neg)

        for _, row in df_neg.iterrows():
            _plot_coef_point(ax_neg, row, feat_to_ypos[row["feature"]])
        for _, row in df_pos.iterrows():
            _plot_coef_point(ax_pos, row, feat_to_ypos[row["feature"]])

        # Y-axis: labels on left panel only
        ax_neg.set_yticks(y_positions)
        ax_neg.set_yticklabels(
            y_labels, fontsize=A1_LABEL_FONT,
            horizontalalignment="right", linespacing=0.92,
        )
        ax_neg.set_ylim(current_y - 0.4, -0.6)
        ax_pos.set_yticks(y_positions)
        plt.setp(ax_pos.get_yticklabels(), visible=False)
        ax_pos.tick_params(axis="y", length=0)

        # X-axis limits: tight to data, small gap around zero
        _margin = 0.05
        ax_neg.set_xlim(
            float(df_neg["std_coef"].min()) - _margin,
            float(df_neg["std_coef"].max()) + _margin,
        )
        ax_pos.set_xlim(
            float(df_pos["std_coef"].min()) - _margin,
            float(df_pos["std_coef"].max()) + _margin,
        )


        # Tick params
        ax_neg.tick_params(axis="y", labelsize=A1_LABEL_FONT, pad=6, length=0)
        ax_neg.tick_params(axis="x", labelsize=A1_XLABEL_FONT - 1)
        ax_pos.tick_params(axis="x", labelsize=A1_XLABEL_FONT - 1)

        # X-axis labels
        ax_neg.set_xlabel(
            "Std age coef (negative)", fontsize=A1_XLABEL_FONT
        )
        ax_pos.set_xlabel(
            "Std age coef (positive)", fontsize=A1_XLABEL_FONT
        )

        # Centered title above both panels
        fig.text(
            0.5, 0.92,
            f"Overall age-associated pathway representations "
            f"({coef_note})\n"
            "Red/blue = positive/negative age coefficient  |  "
            "Filled/open markers = mean gate above/below 1",
            ha="center", va="bottom",
            fontsize=A1_TITLE_FONT,
        )

        # ── Broken axis: add // marks and tighten spacing ──
        if A1_XAXIS_MODE == "broken":
            # Minimize gap between panels
            ax_neg.spines["right"].set_visible(False)
            ax_pos.spines["left"].set_visible(False)
            ax_neg.tick_params(axis="x", which="both",
                               right=False, labelright=False)
            ax_pos.tick_params(axis="x", which="both",
                               left=False, labelleft=False)
            # Re-enable x tick labels on both
            ax_neg.tick_params(axis="x", labelbottom=True)
            ax_pos.tick_params(axis="x", labelbottom=True)

            # Diagonal break marks at the junction
            d = 0.012
            kwargs = dict(
                transform=ax_neg.transAxes,
                color="k", clip_on=False, linewidth=0.8,
            )
            ax_neg.plot(
                (1 - d, 1 + d), (-d, +d), **kwargs
            )
            ax_neg.plot(
                (1 - d, 1 + d), (1 - d, 1 + d), **kwargs
            )
            kwargs.update(transform=ax_pos.transAxes)
            ax_pos.plot(
                (-d, +d), (-d, +d), **kwargs
            )
            ax_pos.plot(
                (-d, +d), (1 - d, 1 + d), **kwargs
            )

    # === A2: 4 representative trajectories ===
    age_min_plot = float(gate_filt["age"].min())
    age_max_plot = float(gate_filt["age"].max())

    for tidx, row in enumerate(traj_rows):
        ax = fig.add_subplot(gs[1, tidx])
        ax.set_box_aspect(0.6)
        feat = row["feature"]
        age_coef = row["age_coef"]

        for tissue in TISSUE_ORDER:
            tdata = gate_filt[gate_filt[GROUP_COL] == tissue]

            if tissue == "Blood" and len(tdata) > BLOOD_SCATTER_MAX:
                tdata_plot = tdata.sample(BLOOD_SCATTER_MAX, random_state=42)
            else:
                tdata_plot = tdata

            ax.scatter(
                tdata_plot["age"], tdata_plot[feat],
                color=TISSUE_COLORS[tissue], alpha=0.2, s=5,
                edgecolors="none",
            )

            mean_age_t = tdata["age"].mean()
            mean_gate_t = tdata[feat].mean()
            intercept_t = mean_gate_t - age_coef * mean_age_t
            x_line = np.array([age_min_plot, age_max_plot])
            ax.plot(
                x_line, intercept_t + age_coef * x_line,
                color=TISSUE_COLORS[tissue], linewidth=1.5, label=tissue,
            )

        ax.set_title(
            f"{short_name(feat, 25)}\n"
            f"\u0394g(20\u219290)={row['delta_g_20_90']:.3f}, "
            f"{format_fdr(row['fdr_p'], label='q')}",
            fontsize=A2_TITLE_FONT, pad=6,
        )
        if tidx == 0:
            ax.set_ylabel("Gate silu(g)", fontsize=A2_LABEL_FONT)
        ax.set_xlabel("Age (years)", fontsize=A2_LABEL_FONT)
        ax.tick_params(labelsize=A2_LABEL_FONT - 2)

    # Shared legend below A2 trajectory row
    _handles_a2, _labels_a2 = ax.get_legend_handles_labels()
    fig.legend(
        _handles_a2, _labels_a2,
        loc="lower center", ncol=len(TISSUE_ORDER),
        fontsize=A2_LEGEND_FONT, framealpha=0.7,
        bbox_to_anchor=(0.5, -0.01),
    )

    fig.suptitle(
        f"Overall age-associated pathway representations at the "
        f"{stage_label} gating layer\n"
        "Colored lines are tissue-specific predictions under a common age "
        "coefficient; vertical separation reflects adjusted tissue intercepts, "
        "not tissue-specific age slopes",
        fontsize=A1_TITLE_FONT + 1, y=1.01,
    )
    fig.subplots_adjust(left=0.0, right=0.98, bottom=0.0, top=0.91)
    plt.savefig(
        f"{OUTPUT_DIR}/rowA_overall_age_associated_{stage_name}.pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"{OUTPUT_DIR}/rowA_overall_age_associated_{stage_name}.svg",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)
    print(
        f"Stage {STAGE_IDX} ({stage_name}): Panel A saved "
        f"({n_sig} FDR-sig, {len(df_plot)} in coef plot, "
        f"{len(traj_rows)} trajectories)"
    )

In [ ]:
# === Figure Row B: Tissue-Dependent Pathway Representations ===
# Top N_INTERACTION gates from Analysis 2 (significant Age:Tissue interaction).
# Scatter: age (x) vs gate (y), colored by tissue.
# Fitted lines: separate tissue trend lines (different slopes from interaction model).

for STAGE_IDX in STAGE_INDICES:
    df1, gate_filt, feature_cols, stage_name = analysis1[STAGE_IDX]
    df2 = analysis2[STAGE_IDX]
    top_feats = df2.head(N_INTERACTION)["feature"].tolist()

    n_cols = N_INTERACTION
    fig, axes = plt.subplots(
        1, n_cols, figsize=(n_cols * 4.5, 4.0), squeeze=False
    )

    for idx, feat in enumerate(top_feats):
        ax = axes[0, idx]
        row = df2[df2["feature"] == feat].iloc[0]

        for tissue in TISSUE_ORDER:
            tdata = gate_filt[gate_filt[GROUP_COL] == tissue]

            if tissue == "Blood" and len(tdata) > BLOOD_SCATTER_MAX:
                tdata_plot = tdata.sample(BLOOD_SCATTER_MAX, random_state=42)
            else:
                tdata_plot = tdata

            ax.scatter(
                tdata_plot["age"], tdata_plot[feat],
                color=TISSUE_COLORS[tissue], alpha=0.2, s=5, edgecolors="none",
            )

            # Tissue-specific slope from interaction model
            slope = row[f"{tissue}_slope"]
            mean_age_t = tdata["age"].mean()
            mean_gate_t = tdata[feat].mean()
            intercept_t = mean_gate_t - slope * mean_age_t
            x_line = np.array([gate_filt["age"].min(), gate_filt["age"].max()])
            ax.plot(
                x_line, intercept_t + slope * x_line,
                color=TISSUE_COLORS[tissue], linewidth=1.5, label=tissue,
            )

        ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)
        ax.set_title(
            f"{short_name(feat, 25)}\n"
            f"int_FDR={row['fdr_p']:.2e}, "
            f"effect={row['interaction_effect_size']:.5f}",
            fontsize=BASE_FONT - 2,
        )
        if idx == 0:
            ax.set_ylabel("Gate silu(g)", fontsize=BASE_FONT - 1)
        ax.set_xlabel("Age (years)", fontsize=BASE_FONT - 1)
        ax.tick_params(labelsize=BASE_FONT - 2)

    # Shared legend below all subplots
    _handles, _labels = axes[0, -1].get_legend_handles_labels()
    fig.legend(
        _handles, _labels,
        loc="lower center", ncol=len(TISSUE_ORDER),
        fontsize=BASE_FONT - 4, framealpha=0.7,
        bbox_to_anchor=(0.5, -0.01),
    )

    fig.suptitle(
        f"Tissue-Dependent Age Associations ({stage_name})\n"
        f"Gate ~ Age \u00d7 Tissue + Study  |  Top {N_INTERACTION} by interaction FDR \u00d7 effect size  |  "
        f"Diverging slopes = context-dependent modulation",
        fontsize=BASE_FONT + 1, y=1.02,
    )
    plt.tight_layout(rect=(0, 0.05, 1, 0.98))
    plt.savefig(
        f"{OUTPUT_DIR}/rowB_tissue_dependent_{stage_name}.pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"{OUTPUT_DIR}/rowB_tissue_dependent_{stage_name}.svg",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)
    print(f"Stage {STAGE_IDX} ({stage_name}): Row B saved")

In [ ]:
# === Summary tables + CSV export ===

for STAGE_IDX in STAGE_INDICES:
    df1, gate_filt, feature_cols, stage_name = analysis1[STAGE_IDX]
    df2 = analysis2[STAGE_IDX]

    # Add pathway names
    df1_out = df1.copy()
    df1_out["pathway_name"] = df1_out["feature"].map(
        lambda f: name_map.get(f, f)
    )
    df2_out = df2.copy()
    df2_out["pathway_name"] = df2_out["feature"].map(
        lambda f: name_map.get(f, f)
    )

    csv1 = f"{OUTPUT_DIR}/analysis1_overall_age_associated_{stage_name}.csv"
    csv2 = f"{OUTPUT_DIR}/analysis2_tissue_dependent_{stage_name}.csv"

    df1_out.to_csv(csv1, index=False)
    df2_out.to_csv(csv2, index=False)

    print(f"\n{'=' * 70}")
    print(f"Stage {STAGE_IDX} ({stage_name})")
    print(f"{'=' * 70}")
    print(f"\nSaved: {csv1}")
    print(f"Saved: {csv2}")

    # Analysis 1 summary
    print(f"\n--- Analysis 1: Top {N_OVERALL} Overall Age-Associated Gates ---")
    print("    Model: Gate ~ Age + Tissue + Study")
    for _, r in df1.head(N_OVERALL).iterrows():
        print(
            f"  {r['rank']}. {r['feature']} ({short_name(r['feature'], 35)})"
        )
        print(
            f"      age_coef={r['age_coef']:.6f}, std_coef={r['std_coef']:.4f}, "
            f"\u0394g(20\u219290)={r['delta_g_20_90']:.4f}, "
            f"t={r['age_t']:.2f}, p={r['age_p']:.2e}, {format_fdr(r['fdr_p'])}"
        )

    # Analysis 2 summary
    print(f"\n--- Analysis 2: Top {N_INTERACTION} Tissue-Dependent Gates ---")
    print("    Model: Gate ~ Age * Tissue + Study  (omnibus interaction F-test)")
    for _, r in df2.head(N_INTERACTION).iterrows():
        print(
            f"  {r['rank']}. {r['feature']} ({short_name(r['feature'], 35)})"
        )
        print(
            f"      omnibus_F={r['omnibus_F']:.2f}, "
            f"p={r['omnibus_p']:.2e}, FDR={r['fdr_p']:.2e}"
        )
        for tissue in TISSUE_ORDER:
            print(
                f"        {tissue}: slope={r[f'{tissue}_slope']:.6f}/yr"
            )
        print(f"      effect_size={r['interaction_effect_size']:.6f}")

print("\n\nDone. All analyses, figures, and CSVs exported.")